# Minerva Evals

## Results folders

In [1]:
from pathlib import Path
import re
import json
import pandas as pd
from pydantic import BaseModel, Field

class CellResult(BaseModel):
    top_k: int
    hybrid_alpha: float

class Metrics(BaseModel):
    recall_at_5: float
    recall_at_10: float
    recall_at_20: float
    success_at_5: int
    success_at_10: int
    success_at_20: int
    mrr_at_10: float

class Hit(BaseModel):
    rank: int
    chunk_id: str
    source_id: str
    score: float
    gold_hit: bool

class EvalResult(BaseModel):
    collection: str          # injected from run.json; not present in the jsonl row
    query_id: str
    query: str
    cell: CellResult
    gold_sources: list[str] = Field(default_factory=list)
    metrics: Metrics
    latency_ms: int
    error: str | None = None
    hits: list[Hit] = Field(default_factory=list)

results_path = Path("../results")

In [2]:
def find_run_dirs(results_path: Path) -> list[Path]:
    return sorted(p for p in results_path.iterdir()
                  if p.is_dir() and re.match(r"\d{4}-\d{2}-\d{2}", p.name))

def load_run(run_dir: Path) -> list[EvalResult]:
    collection = json.loads((run_dir / "run.json").read_text())["collection"]
    lines = (run_dir / "details.jsonl").read_text().splitlines()
    return [EvalResult.model_validate(json.loads(line) | {"collection": collection})
            for line in lines if line.strip()]

In [3]:
all_results = [r for d in find_run_dirs(results_path) for r in load_run(d)]

collections = {r.collection for r in all_results}
print(f"{len(all_results)} rows from {len(collections)} collections: {collections}")

192 rows from 2 collections: {'wikipedia-qwen2-5', 'wikipedia-nollm'}


## Run summary

One row per run folder: collection, timestamp, sweep matrix, query count.

In [4]:
def summarize_run(run_dir: Path) -> dict:
    run = json.loads((run_dir / "run.json").read_text())
    rows = load_run(run_dir)
    matrix = run["resolved_sweep"]["matrix"]
    return {
        "folder": run_dir.name,
        "collection": run["collection"],
        "timestamp": run["timestamp"],
        "sweep": f"top_k={matrix['top_k']} alpha={matrix['hybrid_alpha']}",
        "queries": len({r.query_id for r in rows}),
        "rows": len(rows),
    }

run_summary = pd.DataFrame(summarize_run(d) for d in find_run_dirs(results_path))
run_summary

,folder,collection,timestamp,sweep,queries,rows
0,2026-06-26T15-46-32Z_wikipedia-v1,wikipedia-nollm,2026-06-26T15:46:32Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",32,96
1,2026-06-26T15-47-34Z_wikipedia-v1,wikipedia-qwen2-5,2026-06-26T15:47:34Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",32,96


## Metrics view

Mean over queries, grouped by `(collection, hybrid_alpha)`. Reproduces the
headline table from the sweep-findings spec; styled as a heatmap.

In [5]:
metric_cols = ["R@5", "R@10", "R@20", "MRR@10"]

per_query = pd.DataFrame([
    {
        "collection": r.collection,
        "alpha": r.cell.hybrid_alpha,
        "R@5": r.metrics.recall_at_5,
        "R@10": r.metrics.recall_at_10,
        "R@20": r.metrics.recall_at_20,
        "MRR@10": r.metrics.mrr_at_10,
    }
    for r in all_results
])

groups = per_query.groupby(["collection", "alpha"])
metrics = groups[metric_cols].mean().round(3)
metrics["n"] = groups.size()

metrics.style.background_gradient(cmap="RdYlGn", subset=metric_cols)

## Failure picker

For a chosen `(collection, hybrid_alpha, metric)`, return the suspect rows —
queries scoring below `threshold` on that metric. Default `threshold=1.0` catches
partial misses, not only full zeros. These rows feed the audit view.

In [6]:
def pick_failures(collection: str, alpha: float, metric: str,
                  threshold: float = 1.0) -> list[EvalResult]:
    return [r for r in all_results
            if r.collection == collection
            and r.cell.hybrid_alpha == alpha
            and getattr(r.metrics, metric) < threshold]

# spec example: (wikipedia-nollm, alpha=0.5, recall_at_10)
metric = "recall_at_10"
failures = pick_failures("wikipedia-nollm", 0.5, metric)

print(f"{len(failures)} suspect queries")
pd.DataFrame([
    {"query_id": r.query_id, "query": r.query, metric: getattr(r.metrics, metric)}
    for r in failures
])

2 suspect queries


,query_id,query,recall_at_10
0,Arthropod.md-2,Find the bugs in the tree of life,0.0
1,Shortness_of_breath.md-2,a sensation of inadequate or strained respiration,0.5


## Audit view

For each failing query: the gold article(s) with a corpus snippet, plus the
deduplicated top-10 retrieved articles (rank + GOLD marker). Lets each miss be
classified as a real retrieval failure or a mislabeled gold.

The corpus path is read from `config.local.json` (gitignored, machine-specific).
Copy `config.local.json.example` and set `corpus_dir`.

In [7]:
config = json.loads(Path("config.local.json").read_text())
CORPUS_DIR = Path(config["corpus_dir"])

def read_snippet(source_id: str, max_chars: int = 300) -> str:
    text = (CORPUS_DIR / source_id).read_text()
    body = re.sub(r"^---\n.*?\n---\n", "", text, count=1, flags=re.DOTALL)  # drop frontmatter
    paras = [s for p in body.split("\n\n") if (s := p.strip()) and not s.startswith("#")]
    snippet = paras[0] if paras else ""
    return snippet[:max_chars] + ("…" if len(snippet) > max_chars else "")

def top_articles(hits: list[Hit], k: int = 10) -> list[Hit]:
    seen, out = set(), []
    for h in sorted(hits, key=lambda h: h.rank):
        if h.source_id not in seen:
            seen.add(h.source_id)
            out.append(h)
        if len(out) == k:
            break
    return out

def audit(failures: list[EvalResult]) -> None:
    for r in failures:
        print(f"=== {r.query_id}  recall_at_10={r.metrics.recall_at_10} ===")
        print(f"query: {r.query!r}")
        for g in r.gold_sources:
            print(f"gold  [{g}]: {read_snippet(g)}")
        print("top-10 articles:")
        for h in top_articles(r.hits):
            mark = "GOLD" if h.source_id in r.gold_sources else "    "
            print(f"  {h.rank:>2} {mark} {h.source_id}  ({h.score:.4f})")
        print()

audit(failures)

=== Arthropod.md-2  recall_at_10=0.0 ===
query: 'Find the bugs in the tree of life'
gold  [Arthropod.md]: **Arthropods** (/ˈɑːrθrəˌpɒd/ *AR-thrə-pod*) are [invertebrates](Invertebrate "Invertebrate") in the [phylum](Phylum "Phylum") **Arthropoda**. They possess an exoskeleton with a cuticle made of [chitin](Chitin "Chitin"), often mineralised with [calcium carbonate](Calcium_carbonate "Calcium carbonate…
top-10 articles:
   1      Plant.md  (0.0121)
   2      Frog.md  (0.0118)
   3      Insect.md  (0.0113)
   4      Moorish_architecture.md  (0.0112)
   5      Beetle.md  (0.0110)
   6      Merit_(Buddhism).md  (0.0110)
   8      Meaning_of_life.md  (0.0109)
   9      Fungus.md  (0.0108)
  10      Delonix_regia.md  (0.0108)
  11      Mauna_Kea.md  (0.0108)

=== Shortness_of_breath.md-2  recall_at_10=0.5 ===
query: 'a sensation of inadequate or strained respiration'
gold  [Shortness_of_breath.md]: **Shortness of breath** (**SOB**), known as **dyspnea** (in [AmE](AmE "AmE")) or **dyspnoea*